# 1. Build & push

Generates the SLURM/runner scripts, uploads the project to the cluster, creates
the conda environment, and runs a preflight check.

Run `0_config.ipynb` first — this notebook just loads the settings it saved.

In [ ]:
import nb2slurm
wf, cfg = nb2slurm.load_config("control_config.json")

## Generate the scripts

Renders `scripts/` (runner, `job.slurm`, the submitters) and `environment.yml`
into the project root, and writes `jobs.txt` from `jobs.json`.

In [ ]:
for kind, path in wf.build().items():
    print(f"{kind:13s} -> {path}")

## Push the project to the cluster

Uploads source only (notebooks/, scripts/, jobs.json, ...). It **excludes**
`runs/` and `done/`, so pushing can never wipe results already on the cluster.
(Requires `rsync` on your machine.)

In [ ]:
wf.push(ssh=cfg, dry_run=True)   # preview the rsync command
wf.push(ssh=cfg)                 # for real

## Create the conda environment (one-time, optional)

Skip if the cluster provides your environment.

In [ ]:
wf.create_environment(ssh=cfg)

## Preflight check

Confirm the cluster is ready before submitting.

In [ ]:
wf.check(ssh=cfg)